In [2]:
import pandas as pd
import numpy as np
import yfinance as yf


In [3]:
tickers = [
    "^NDX",  # Indeks Nasdaq 100 (jako benchmark rynkowy)
    "AAPL",  # Apple
    "MSFT",  # Microsoft
    "NVDA",  # NVIDIA
    "AMZN",  # Amazon
    "GOOGL", # Alphabet 
    "META",  # Meta
    "TSLA",  # Tesla
    "AVGO",  # Broadcom
    "COST",  # Costco Wholesale
    "NFLX",  # Netflix
    "AMD",   # Advanced Micro Devices 
    "PEP",   # PepsiCo 
]

prices = yf.download(tickers=tickers, start ="2010-01-01",multi_level_index=False, auto_adjust=True)
prices = prices["Close"]

prices.isnull().sum()
prices.info()


[*********************100%***********************]  13 of 13 completed

<class 'pandas.DataFrame'>
DatetimeIndex: 4229 entries, 2010-01-04 to 2026-10-26
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    4229 non-null   float64
 1   AMD     4229 non-null   float64
 2   AMZN    4229 non-null   float64
 3   AVGO    4229 non-null   float64
 4   COST    4229 non-null   float64
 5   GOOGL   4229 non-null   float64
 6   META    3630 non-null   float64
 7   MSFT    4229 non-null   float64
 8   NFLX    4229 non-null   float64
 9   NVDA    4229 non-null   float64
 10  PEP     4229 non-null   float64
 11  TSLA    4107 non-null   float64
 12  ^NDX    4229 non-null   float64
dtypes: float64(13)
memory usage: 462.5 KB


In [4]:
#R dziennie stope zwrotu
daily_return = prices.pct_change(1)
daily_return = daily_return.iloc[1:]

#ogolny zwrot
total_return = (1+daily_return).prod() - 1


#Zannualizowana zmiennosc / wahania
annualized_volatility= daily_return.std() * (252**0.5)
annualized_volatility


#zannualizowany zwrot
years = len(daily_return) /252
annualized_return = (1 + total_return) ** (1/years) -1
annualized_return

#Wskaznik sharpa
risk_free_rate = 0
sharpe_ratio = ((annualized_return/years) - risk_free_rate)/annualized_volatility
sharpe_ratio

#najwiekszy dołek
rolling_max = prices.cummax()
drawdown = (prices - rolling_max) / rolling_max
drawdown = drawdown.min()

market = daily_return["^NDX"].values
scl= {}

#scl i korelacja
for ticker in daily_return.columns:

    company_return = daily_return[ticker].values
    mask = ~np.isnan(company_return) & ~np.isnan(market)

    b, a = np.polyfit(market[mask], company_return[mask], 1)
    corr = np.corrcoef(market[mask],company_return[mask])

    scl[ticker] = {"alfa":a, "beta": b, "corr" : corr[1,0]}
 
scl = pd.DataFrame(scl).T




In [5]:
summary_table = pd.DataFrame({
    "total_return" : total_return,
    "annualized_volatility" : annualized_volatility,
    "sharpe_ratio" : sharpe_ratio,
    "max_drawdown" : drawdown,
    "scl_alfa": scl["alfa"],
    "scl_beta": scl["beta"],
    "corr" : scl["corr"]
    })

summary_table.to_csv("summary_table.csv")
daily_return.to_csv("daily_return.csv")
prices.to_csv("prices.csv")